# Train ChemProp on F2 (similarity split)

Predict F2 docking score from SMILES with **ChemProp** (graph MPNN).

- **Label**: F2 docking score (kcal/mol; lower = better binding)
- **Split**: ChemProp-style **Morgan / Tanimoto similarity split** (radius=2, threshold=0.7) — *not* ChemProp's default random split
- **Morgan fingerprints**: used for the split (and optional XGBoost baseline only); ChemProp takes SMILES graphs as input
- **Env**: ChemProp 2.x needs Python ≥3.11 — use kernel **Python (chemprop)** / `micromamba activate chemprop` (dockstring env is 3.10). This machine uses **chemprop 2.1.2** (2.3.x pulls a broken `cuik_molmaker` wheel on this macOS).

On macOS, if OpenMP complains about duplicate libs, set `KMP_DUPLICATE_LIB_OK=TRUE` before starting the kernel.


In [1]:
import os

# Harmless on most setups; avoids OpenMP abort when conda+pip mix libomp on macOS.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pathlib import Path
import json

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATASET_PATH = Path("./data/dockstring-dataset.tsv")
OUT_DIR = Path("./data/chemprop_f2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MORGAN_RADIUS = 2
MORGAN_NBITS = 2048
TANIMOTO_THRESHOLD = 0.7
TEST_FRAC = 0.2
VAL_FRAC_OF_TRAIN = 0.1  # random carve from similarity-train pool for early stopping
N_SAMPLE = 25_000  # raise toward full ~260k on SAVIO / GPU
SEED = 0
MAX_EPOCHS = 20
BATCH_SIZE = 64
NUM_WORKERS = 0

assert DATASET_PATH.exists(), f"Missing {DATASET_PATH}"
print(f"out dir: {OUT_DIR.resolve()}")


out dir: /Users/nathanye/Documents/GitHub/merck-simple-experiment/data/chemprop_f2


## 1. Load F2 & SMILES hygiene

Standardize → neutralize → canonical SMILES, drop missing F2, dedupe on standardized SMILES.

In [2]:
df = pd.read_csv(DATASET_PATH, sep="\t")
f2 = df[["inchikey", "smiles", "F2"]].rename(columns={"F2": "docking_score"}).copy()
print(f"raw rows: {len(f2):,}  missing F2: {int(f2['docking_score'].isna().sum()):,}")

uncharger = rdMolStandardize.Uncharger()
lfc = rdMolStandardize.LargestFragmentChooser()


def standardize_smiles(smi: str):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mol = lfc.choose(mol)
    mol = uncharger.uncharge(mol)
    return Chem.MolToSmiles(mol)


clean = f2.dropna(subset=["docking_score"]).copy()
if N_SAMPLE is not None and N_SAMPLE < len(clean):
    clean = clean.sample(n=N_SAMPLE, random_state=SEED)

clean["smiles"] = clean["smiles"].map(standardize_smiles)
clean = (
    clean.dropna(subset=["smiles"])
    .drop_duplicates("smiles")
    .reset_index(drop=True)
)
print(f"after standardize + dedupe: {len(clean):,}")

# ChemProp CSV: smiles + target column named F2
chemprop_df = clean[["smiles", "docking_score"]].rename(columns={"docking_score": "F2"})
full_csv = OUT_DIR / "f2_all.csv"
chemprop_df.to_csv(full_csv, index=False)
print(f"wrote {full_csv}  ({len(chemprop_df):,} rows)")
chemprop_df.head()

raw rows: 260,155  missing F2: 5


[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49:15] Running LargestFragmentChooser
[23:49:15] Running Uncharger
[23:49

after standardize + dedupe: 24,998
wrote data/chemprop_f2/f2_all.csv  (24,998 rows)


[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49:26] Running LargestFragmentChooser
[23:49:26] Running Uncharger
[23:49

,smiles,F2
0,CN(C)Cc1ccccc1-c1ccc(-c2cnc3c(C(F)(F)F)nn(-c4c...,-9.8
1,O=C(Cc1coc2ccc3ccccc3c12)Nc1nnc(C2CC2)s1,-9.5
2,Brc1cccc(-c2nn(-c3ccccc3)c3c2cnc2cc4c(cc23)OCO...,-10.0
3,COC(=O)c1c(OCCN2CCOCC2)c2ccccc2c2oc3c(c12)C(=O...,-8.6
4,O=C(O)C1=C(C(=O)Nc2ccc3c(c2)CCC3)CCCC1,-8.0


## 2. Similarity split (Morgan radius=2, Tanimoto 0.7)

Same ChemProp-style procedure as `eda.ipynb`: train gets a random majority slice; remaining molecules enter **test only if** max Tanimoto to train is **< 0.7**, else they stay in train.

Validation is a **random** carve from the train pool (for early stopping only) — test stays similarity-held-out.

In [3]:
from rdkit.Chem import rdFingerprintGenerator

MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(
    radius=MORGAN_RADIUS, fpSize=MORGAN_NBITS
)


def morgan_fp(smi: str):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return MORGAN_GEN.GetFingerprint(mol)


def similarity_split(fps, test_frac=0.2, threshold=0.7, seed=0):
    """ChemProp-style fingerprint similarity split (train vs test)."""
    rng = np.random.default_rng(seed)
    n = len(fps)
    order = rng.permutation(n).tolist()
    n_train_target = int(round(n * (1.0 - test_frac)))

    train = order[:n_train_target]
    remain = order[n_train_target:]
    train_fps = [fps[i] for i in train]

    test, extra_train = [], []
    for i in remain:
        max_sim = max(DataStructs.BulkTanimotoSimilarity(fps[i], train_fps))
        if max_sim < threshold:
            test.append(i)
        else:
            extra_train.append(i)
    train = train + extra_train
    return np.array(train), np.array(test)


fps = [morgan_fp(s) for s in chemprop_df["smiles"]]
ok = [fp is not None for fp in fps]
if not all(ok):
    chemprop_df = chemprop_df.loc[ok].reset_index(drop=True)
    fps = [fp for fp, keep in zip(fps, ok) if keep]
    chemprop_df.to_csv(full_csv, index=False)

train_pool_idx, test_idx = similarity_split(
    fps, test_frac=TEST_FRAC, threshold=TANIMOTO_THRESHOLD, seed=SEED
)
if len(test_idx) == 0:
    raise RuntimeError("Empty test set — lower threshold or change sample/seed")

rng = np.random.default_rng(SEED)
perm = rng.permutation(train_pool_idx)
n_val = max(1, int(round(len(perm) * VAL_FRAC_OF_TRAIN)))
val_idx = perm[:n_val]
train_idx = perm[n_val:]

print(
    f"split: train={len(train_idx):,}  val={len(val_idx):,}  test={len(test_idx):,}  "
    f"(requested test_frac={TEST_FRAC}, threshold={TANIMOTO_THRESHOLD})"
)
print(
    f"split params: Morgan radius={MORGAN_RADIUS}, nBits={MORGAN_NBITS}, "
    f"Tanimoto threshold={TANIMOTO_THRESHOLD}"
)

check_n = min(200, len(test_idx))
train_pool_fps = [fps[i] for i in train_pool_idx]
max_sims = [
    max(DataStructs.BulkTanimotoSimilarity(fps[i], train_pool_fps))
    for i in test_idx[:check_n]
]
print(
    f"max Tanimoto(test→train_pool) on {check_n} test mols: "
    f"max={np.max(max_sims):.3f}, mean={np.mean(max_sims):.3f} "
    f"(should be < {TANIMOTO_THRESHOLD})"
)


split: train=18,734  val=2,082  test=4,182  (requested test_frac=0.2, threshold=0.7)
split params: Morgan radius=2, nBits=2048, Tanimoto threshold=0.7
max Tanimoto(test→train_pool) on 200 test mols: max=0.821, mean=0.511 (should be < 0.7)


In [4]:
train_csv = OUT_DIR / "train.csv"
val_csv = OUT_DIR / "val.csv"
test_csv = OUT_DIR / "test.csv"
splits_json = OUT_DIR / "similarity_split_indices.json"

chemprop_df.iloc[train_idx].to_csv(train_csv, index=False)
chemprop_df.iloc[val_idx].to_csv(val_csv, index=False)
chemprop_df.iloc[test_idx].to_csv(test_csv, index=False)

import json

split_payload = {
    "method": "morgan_tanimoto_similarity",
    "morgan_radius": MORGAN_RADIUS,
    "morgan_nbits": MORGAN_NBITS,
    "tanimoto_threshold": TANIMOTO_THRESHOLD,
    "test_frac_requested": TEST_FRAC,
    "val_frac_of_train": VAL_FRAC_OF_TRAIN,
    "seed": SEED,
    "n_sample": N_SAMPLE,
    "train_idx": train_idx.tolist(),
    "val_idx": val_idx.tolist(),
    "test_idx": test_idx.tolist(),
}
splits_json.write_text(json.dumps(split_payload))
print(f"wrote {train_csv.name}, {val_csv.name}, {test_csv.name}, {splits_json.name}")

wrote train.csv, val.csv, test.csv, similarity_split_indices.json


## 3. Optional XGBoost baseline (Morgan fingerprints)

Fast sanity check on the **same** similarity split. Skip if you only want ChemProp.

In [5]:
RUN_XGB_BASELINE = True

if RUN_XGB_BASELINE:
    from xgboost import XGBRegressor

    X = np.vstack([np.asarray(fp) for fp in fps])
    y = chemprop_df["F2"].to_numpy(dtype=float)

    # Train+val together for the tree baseline (val was only for ChemProp early stopping)
    fit_idx = np.concatenate([train_idx, val_idx])
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=-1,
    )
    xgb.fit(X[fit_idx], y[fit_idx])
    pred = xgb.predict(X[test_idx])
    y_te = y[test_idx]
    rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
    mae = float(mean_absolute_error(y_te, pred))
    r2 = float(r2_score(y_te, pred))
    print("XGBoost (Morgan) test metrics")
    print(f"  RMSE={rmse:.4f}  MAE={mae:.4f}  R2={r2:.4f}")
    print(
        f"  split=Morgan/Tanimoto similarity, radius={MORGAN_RADIUS}, "
        f"threshold={TANIMOTO_THRESHOLD}, nBits={MORGAN_NBITS}"
    )
else:
    print("skipped XGBoost baseline")

XGBoost (Morgan) test metrics
  RMSE=0.5903  MAE=0.4632  R2=0.6289
  split=Morgan/Tanimoto similarity, radius=2, threshold=0.7, nBits=2048


## 4. Train ChemProp (custom split indices)

Uses ChemProp's Python API with our similarity-split indices — **not** `make_split_indices(..., "random")`.

In [6]:
from lightning import pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from chemprop import data, featurizers, models, nn

smis = chemprop_df["smiles"].tolist()
ys = chemprop_df[["F2"]].to_numpy(dtype=float)

all_data = [
    data.MoleculeDatapoint.from_smi(smi, y)
    for smi, y in zip(smis, ys)
]

# Custom indices: wrap each as a 1-replicate list for split_data_by_indices
train_data, val_data, test_data = data.split_data_by_indices(
    all_data,
    [train_idx.tolist()],
    [val_idx.tolist()],
    [test_idx.tolist()],
)

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
train_dset = data.MoleculeDataset(train_data[0], featurizer)
scaler = train_dset.normalize_targets()

val_dset = data.MoleculeDataset(val_data[0], featurizer)
val_dset.normalize_targets(scaler)

test_dset = data.MoleculeDataset(test_data[0], featurizer)
# leave test unscaled; predictor UnscaleTransform maps back to raw F2 units

train_loader = data.build_dataloader(
    train_dset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
val_loader = data.build_dataloader(
    val_dset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False
)
test_loader = data.build_dataloader(
    test_dset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False
)

print(
    f"ChemProp datasets: train={len(train_dset):,}  val={len(val_dset):,}  "
    f"test={len(test_dset):,}"
)

ChemProp datasets: train=18,734  val=2,082  test=4,182


In [ ]:
mp = nn.BondMessagePassing()
agg = nn.MeanAggregation()
output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)
ffn = nn.RegressionFFN(output_transform=output_transform)
metric_list = [nn.metrics.RMSE(), nn.metrics.MAE()]
mpnn = models.MPNN(mp, agg, ffn, batch_norm=True, metrics=metric_list)

ckpt_dir = OUT_DIR / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

checkpointing = ModelCheckpoint(
    dirpath=str(ckpt_dir),
    filename="best-{epoch}-{val_loss:.3f}",
    monitor="val_loss",
    mode="min",
    save_last=True,
)
early_stop = EarlyStopping(monitor="val_loss", mode="min", patience=5)

trainer = pl.Trainer(
    logger=False,
    enable_checkpointing=True,
    enable_progress_bar=True,
    accelerator="auto",
    devices=1,
    max_epochs=MAX_EPOCHS,
    callbacks=[checkpointing, early_stop],
)

trainer.fit(mpnn, train_loader, val_loader)
print(f"best checkpoint: {checkpointing.best_model_path}")

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.
/Users/nathanye/micromamba/envs/chemprop/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/nathanye/micromamba/envs/chemprop/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1.276                                                                      
Modules in train mode: 25                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/nathanye/micromamba/envs/chemprop/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/nathanye/micromamba/envs/chemprop/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_con
nector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

## 5. Test metrics

Report RMSE / MAE / R² on the similarity-held-out test set, with split fingerprint settings stated explicitly.

In [ ]:
ckpt_path = checkpointing.best_model_path or checkpointing.last_model_path
pred_batches = trainer.predict(
    mpnn, dataloaders=test_loader, ckpt_path=ckpt_path, weights_only=False
)
# each batch is a tensor of predictions in original (unscaled) units
def _batch_to_np(p):
    if hasattr(p, "detach"):
        return p.detach().cpu().numpy().reshape(-1)
    if isinstance(p, (list, tuple)):
        return np.concatenate([_batch_to_np(x) for x in p])
    return np.asarray(p).reshape(-1)

y_pred = np.concatenate([_batch_to_np(p) for p in pred_batches])
y_true = chemprop_df.iloc[test_idx]["F2"].to_numpy(dtype=float)

assert len(y_pred) == len(y_true), (len(y_pred), len(y_true))

rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
mae = float(mean_absolute_error(y_true, y_pred))
r2 = float(r2_score(y_true, y_pred))

print("ChemProp test metrics")
print(f"  RMSE={rmse:.4f}  MAE={mae:.4f}  R2={r2:.4f}")
print(
    f"  split=Morgan/Tanimoto similarity, radius={MORGAN_RADIUS}, "
    f"threshold={TANIMOTO_THRESHOLD}, nBits={MORGAN_NBITS}"
)
print(
    f"  n_train={len(train_idx):,}  n_val={len(val_idx):,}  n_test={len(test_idx):,}  "
    f"n_sample={N_SAMPLE}  epochs≤{MAX_EPOCHS}"
)

metrics_path = OUT_DIR / "test_metrics.json"
metrics_path.write_text(
    json.dumps(
        {
            "model": "chemprop_mpnn",
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "split": {
                "method": "morgan_tanimoto_similarity",
                "morgan_radius": MORGAN_RADIUS,
                "morgan_nbits": MORGAN_NBITS,
                "tanimoto_threshold": TANIMOTO_THRESHOLD,
            },
            "n_train": int(len(train_idx)),
            "n_val": int(len(val_idx)),
            "n_test": int(len(test_idx)),
            "n_sample": N_SAMPLE,
            "checkpoint": str(ckpt_path),
        },
        indent=2,
    )
)
print(f"wrote {metrics_path}")


### Notes / pitfalls

- Metrics above are on a **similarity-held-out** test set; random-split numbers would look better and overstate generalization.
- `N_SAMPLE=25_000` matches the EDA modeling sample; bump toward full DOCKSTRING on SAVIO/GPU.
- Docking scores are an imperfect binding proxy — treat ChemProp as a filter before docking interesting SMILES with `dock_f2.py`.
- Activate kernel: `micromamba activate chemprop` (Python 3.11 + ChemProp 2.x).